In [1]:
!pip install tabicl[all]

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 33.5 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.7/8.7 MB 164.1 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.5/206.5 kB 26.5 MB/s eta 0:00:00
  Attempting uninstall: matplotlib
    Found existing installation: matplotlib 3.10.0
    Uninstalling matplotlib-3.10.0:
      Successfully uninstalled matplotlib-3.10.0


In [2]:
#!/usr/bin/env python3
"""
Train a TabICLv2 Regressor on Moltbook data.
Uses the official, pip-installable `tabicl` library.
"""

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
import matplotlib.pyplot as plt
import joblib
import warnings
from sklearn.inspection import permutation_importance

# Import the TabICLv2 regressor
from tabicl import TabICLRegressor

warnings.filterwarnings('ignore')

# ==========================================
# CONFIGURATION (Identical to your RF script)
# ==========================================
from google.colab import drive
drive.mount('/content/drive')
DESTINATION_DIR = '/content/drive/MyDrive/Colab Notebooks/data/cds'
DATA_PATH = f"{DESTINATION_DIR}/reddit_11_4_full.pkl"

EMBEDDING_COL = "embeddings"
TARGET_COL = "score"
RANDOM_STATE = 42
TEST_SIZE = 0.30
VAL_SIZE_FROM_TEMP = 0.50
DROP_COLS = ['forum','created_utc_dt','title', 'selftext','safe_content',"content"]

# ==========================================
# 1. Load and prepare data (SAME AS YOUR SCRIPT)
# ==========================================
print("Loading data...")
moltbook = pd.read_pickle(DATA_PATH)
print(f"Original shape: {moltbook.shape}")

# Expand embeddings (SAME AS YOUR SCRIPT)
embedding_lists = moltbook[EMBEDDING_COL].values
lengths = [len(lst) for lst in embedding_lists]
if len(set(lengths)) != 1:
    raise ValueError("Embedding lists have varying lengths.")
emb_dim = lengths[0]
print(f"Embedding dimension: {emb_dim}")

emb_df = pd.DataFrame(
    np.vstack(embedding_lists),
    index=moltbook.index,
    columns=[f"emb_{i}" for i in range(emb_dim)]
)

# Base features and target (SAME AS YOUR SCRIPT)
X_base = moltbook.drop(columns=[TARGET_COL, EMBEDDING_COL] + DROP_COLS)
y_raw = moltbook[TARGET_COL].clip(lower=0)
y = np.log1p(y_raw)  # log-transform target

# Combine features
X_full = pd.concat([X_base, emb_df], axis=1)
print(f"Total features: {X_full.shape[1]}")

# Train/val/test split (SAME AS YOUR SCRIPT)
X_train, X_temp, y_train, y_temp = train_test_split(
    X_full, y, test_size=TEST_SIZE, random_state=RANDOM_STATE
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=VAL_SIZE_FROM_TEMP, random_state=RANDOM_STATE
)

print(f"Training set: {X_train.shape}")
print(f"Validation set: {X_val.shape}")
print(f"Test set: {X_test.shape}")

# ==========================================
# 2. Train TabICLv2 Regressor
# ==========================================
print("\nInitializing TabICLv2 Regressor...")
# The model will download a pre-trained checkpoint on the first use.
# You can adjust the number of ensemble members for speed/performance:
#   - n_estimators: Number of ensemble members (default 32)[reference:5]
#   - batch_size: Ensemble members per batch (default 8)[reference:6]
model = TabICLRegressor(
    n_estimators=16,           # Lower than default for faster training
    batch_size=2,              # Default batch size
    random_state=RANDOM_STATE  # For reproducibility
)

print("Starting TabICLv2 training (this may download a checkpoint on the first run)...")
model.fit(X_train, y_train)    # This performs both 'fit' and 'predict' conceptually

# ==========================================
# 3. Evaluate on test set (R²)
# ==========================================
print("\nEvaluating on test set...")
y_test_pred = model.predict(X_test)
r2_test = r2_score(y_test, y_test_pred)

# Also evaluate on validation set for completeness
y_val_pred = model.predict(X_val)
r2_val = r2_score(y_val, y_val_pred)

print("\n" + "="*50)
print("FINAL RESULTS")
print("="*50)
print(f"Validation R²: {r2_val:.4f}")
print(f"Test R²: {r2_test:.4f}")

# ==========================================
# 4. Feature Importance (Permutation)
# ==========================================
print("\nCalculating permutation importance (this may take a few minutes)...")
def predict_wrapper(model, X):
    return model.predict(X)


Mounted at /content/drive
Loading data...
Original shape: (34867, 23)
Embedding dimension: 768
Total features: 783
Training set: (24406, 783)
Validation set: (5230, 783)
Test set: (5231, 783)

Initializing TabICLv2 Regressor...
Starting TabICLv2 training (this may download a checkpoint on the first run)...
Checkpoint 'tabicl-regressor-v2-20260212.ckpt' not cached.



tabicl-regressor-v2-20260212.ckpt:   0%|          | 0.00/114M [00:00<?, ?B/s]


Evaluating on test set...

FINAL RESULTS
Validation R²: 0.4603
Test R²: 0.4463

Calculating permutation importance (this may take a few minutes)...


In [4]:
print("\nSaving model...")
# Save the model using joblib (recommended for scikit-learn compatible models)
model_filename = f"{DESTINATION_DIR}/tabicl_model_{RANDOM_STATE}.joblib"
joblib.dump(model, model_filename)
print(f"Model saved to: {model_filename}")


Saving model...
Model saved to: /content/drive/MyDrive/Colab Notebooks/data/cds/tabicl_model_42.joblib


In [3]:
print(xxxxxxx)

NameError: name 'xxxxxxx' is not defined

In [ ]:
def safe_predict(model, X, chunk_size=200, description="Predicting"):
    """Memory-safe prediction with chunking"""
    import gc
    n_samples = len(X)
    all_predictions = []
    for start_idx in range(0, n_samples, chunk_size):
        end_idx = min(start_idx + chunk_size, n_samples)
        chunk = X.iloc[start_idx:end_idx]
        chunk_pred = model.predict(chunk)
        all_predictions.append(chunk_pred)
        gc.collect()
    return np.concatenate(all_predictions)

In [ ]:
# ==========================================
# 4. Feature Importance (Only non-embedding, full data, memory-safe)
# ==========================================
print("\nCalculating permutation importance for non-embedding features only...")

# Identify which columns are embeddings
embedding_cols = [col for col in X_train.columns if col.startswith('emb_')]
non_embedding_cols = [col for col in X_train.columns if not col.startswith('emb_')]
print(f"Total features: {len(X_train.columns)}")
print(f"Embedding features: {len(embedding_cols)} (excluded from importance)")
print(f"Non-embedding features: {len(non_embedding_cols)} (will compute importance)")

# Use full validation data (no subset)
X_val_full = X_val
y_val_full = y_val

# Memory-safe permutation importance for selected features
def safe_permutation_importance_subset(model, X, y, feature_cols, n_repeats=3, chunk_size=200):
    """Compute permutation importance only for specified feature columns."""
    from sklearn.metrics import r2_score
    import numpy as np
    import pandas as pd

    # Baseline prediction (using chunked predict)
    print("  Computing baseline R²...")
    y_pred_base = safe_predict(model, X, chunk_size=chunk_size, description="Baseline")
    base_r2 = r2_score(y, y_pred_base)
    print(f"  Baseline R²: {base_r2:.4f}")

    importances = []
    rng = np.random.RandomState(RANDOM_STATE)

    for idx, col in enumerate(feature_cols):
        col_importances = []
        for repeat in range(n_repeats):
            # Permute the column
            X_perm = X.copy()
            X_perm[col] = rng.permutation(X_perm[col].values)
            # Predict permuted data
            y_pred_perm = safe_predict(model, X_perm, chunk_size=chunk_size,
                                      description=f"Feature {idx+1}/{len(feature_cols)} (rep {repeat+1})")
            r2_perm = r2_score(y, y_pred_perm)
            col_importances.append(r2_perm)

        # Importance = drop in R²
        imp = base_r2 - np.mean(col_importances)
        importances.append(imp)

        if (idx+1) % 10 == 0:
            print(f"  Processed {idx+1}/{len(feature_cols)} features")

    return importances

# Run importance only on non-embedding features
n_repeats = 3  # Reduced from 5 to save time/memory
print(f"Using {n_repeats} repeats, full validation set ({len(X_val_full)} samples)")

# Compute importances (this may take a while, but won't crash)
importances = safe_permutation_importance_subset(
    model, X_val_full, y_val_full,
    feature_cols=non_embedding_cols,
    n_repeats=n_repeats,
    chunk_size=200  # Adjust if needed
)

# Create results dataframe
imp_df = pd.DataFrame({
    'Feature': non_embedding_cols,
    'Importance': importances,
    'StdDev': 0  # Optional: you could compute std if desired
}).sort_values('Importance', ascending=False).head(20)

# Plot feature importance
plt.figure(figsize=(10, 12))
plt.barh(imp_df['Feature'][::-1], imp_df['Importance'][::-1],
         color='steelblue', capsize=3)
plt.xlabel('Permutation Importance (Drop in R²)')
plt.title('Top 20 Non-Embedding Features - TabICLv2')
plt.tight_layout()
plt.show()

print(f"Importance calculated for {len(non_embedding_cols)} features")


Calculating permutation importance for non-embedding features only...
Total features: 783
Embedding features: 768 (excluded from importance)
Non-embedding features: 15 (will compute importance)
Using 3 repeats, full validation set (3043 samples)
  Computing baseline R²...


KeyboardInterrupt: 

In [ ]:

DESTINATION_DIR = '/content/drive/MyDrive/Colab Notebooks/data/cds'
DATA_PATH = f"{DESTINATION_DIR}/processed_v1_5_4_new_full.pkl"
EMBEDDING_COL = "embeddings"
TARGET_COL = "score"
RANDOM_STATE = 42
TEST_SIZE = 0.30
VAL_SIZE_FROM_TEMP = 0.50
DROP_COLS = ["safe_content", "content", "id"]

# ==========================================
# 1. Load and prepare data (SAME AS YOUR SCRIPT)
# ==========================================
print("Loading data...")
moltbook = pd.read_pickle(DATA_PATH)
print(f"Original shape: {moltbook.shape}")

# Expand embeddings (SAME AS YOUR SCRIPT)
embedding_lists = moltbook[EMBEDDING_COL].values
lengths = [len(lst) for lst in embedding_lists]
if len(set(lengths)) != 1:
    raise ValueError("Embedding lists have varying lengths.")
emb_dim = lengths[0]
print(f"Embedding dimension: {emb_dim}")

emb_df = pd.DataFrame(
    np.vstack(embedding_lists),
    index=moltbook.index,
    columns=[f"emb_{i}" for i in range(emb_dim)]
)

# Base features and target (SAME AS YOUR SCRIPT)
X_base = moltbook.drop(columns=[TARGET_COL, EMBEDDING_COL] + DROP_COLS)
y_raw = moltbook[TARGET_COL].clip(lower=0)
y = np.log1p(y_raw)  # log-transform target

# Combine features
X_full = pd.concat([X_base, emb_df], axis=1)
print(f"Total features: {X_full.shape[1]}")

# Train/val/test split (SAME AS YOUR SCRIPT)
X_train, X_temp, y_train, y_temp = train_test_split(
    X_full, y, test_size=TEST_SIZE, random_state=RANDOM_STATE
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=VAL_SIZE_FROM_TEMP, random_state=RANDOM_STATE
)

print(f"Training set: {X_train.shape}")
print(f"Validation set: {X_val.shape}")
print(f"Test set: {X_test.shape}")

# ==========================================
# 2. Train TabICLv2 Regressor
# ==========================================
print("\nInitializing TabICLv2 Regressor...")
# The model will download a pre-trained checkpoint on the first use.
# You can adjust the number of ensemble members for speed/performance:
#   - n_estimators: Number of ensemble members (default 32)[reference:5]
#   - batch_size: Ensemble members per batch (default 8)[reference:6]
model = TabICLRegressor(
    n_estimators=16,           # Lower than default for faster training
    batch_size=2,              # Default batch size
    random_state=RANDOM_STATE  # For reproducibility
)

print("Starting TabICLv2 training (this may download a checkpoint on the first run)...")
model.fit(X_train, y_train)    # This performs both 'fit' and 'predict' conceptually

# ==========================================
# 3. Evaluate on test set (R²)
# ==========================================
print("\nEvaluating on test set...")
y_test_pred = model.predict(X_test)
r2_test = r2_score(y_test, y_test_pred)

# Also evaluate on validation set for completeness
y_val_pred = model.predict(X_val)
r2_val = r2_score(y_val, y_val_pred)

print("\n" + "="*50)
print("FINAL RESULTS")
print("="*50)
print(f"Validation R²: {r2_val:.4f}")
print(f"Test R²: {r2_test:.4f}")
